# **Project Title:** Diabetes Prediction Analysis

# **Course:** CSC 405-Data Science Spring 2026

# **Team Members:** Adrian Aldridge, Eman Dweik, Sarah Robinson

# **Dataset Source:**
## https://www.kaggle.com/datasets/miadul/synthetic-diabetes-prediction-dataset




In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv('diabetes_clean.csv')

X = df.drop('diabetes', axis=1)
y = df['diabetes']

categorical_cols = ['gender', 'physical_activity']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Helper function to evaluate classification models
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
    
    print(f"\n{model_name}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    if roc is not None:
        print(f"ROC-AUC:   {roc:.4f}")
    return acc, prec, rec, f1, roc

### **Baseline Models**

In [6]:
# 1. Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_processed, y_train)
evaluate_model(lr, X_test_processed, y_test, "Logistic Regression")

# 2. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_processed, y_train)
evaluate_model(dt, X_test_processed, y_test, "Decision Tree")

# 3. Lin Reg
lin_reg = LinearRegression()
lin_reg.fit(X_train_processed, y_train)
y_pred_lin = np.round(lin_reg.predict(X_test_processed)).clip(0, 1)
y_proba_lin = lin_reg.predict(X_test_processed)  # used as proxy for ROC-AUC
acc_lin = accuracy_score(y_test, y_pred_lin)
prec_lin = precision_score(y_test, y_pred_lin)
rec_lin = recall_score(y_test, y_pred_lin)
f1_lin = f1_score(y_test, y_pred_lin)
roc_lin = roc_auc_score(y_test, y_proba_lin)
print("\nLinear Regression (rounded predictions)")
print(f"Accuracy:  {acc_lin:.4f}")
print(f"Precision: {prec_lin:.4f}")
print(f"Recall:    {rec_lin:.4f}")
print(f"F1-score:  {f1_lin:.4f}")
print(f"ROC-AUC:   {roc_lin:.4f}")



Logistic Regression
Accuracy:  0.4975
Precision: 0.4917
Recall:    0.4168
F1-score:  0.4511
ROC-AUC:   0.5014

Decision Tree
Accuracy:  0.4785
Precision: 0.4731
Recall:    0.4622
F1-score:  0.4676
ROC-AUC:   0.4784

Linear Regression (rounded predictions)
Accuracy:  0.4965
Precision: 0.4905
Recall:    0.4188
F1-score:  0.4518
ROC-AUC:   0.5015


### **Advanced Model (Random Forest)**

In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_processed, y_train)
evaluate_model(rf, X_test_processed, y_test, "Random Forest")


Random Forest
Accuracy:  0.5205
Precision: 0.5181
Recall:    0.4632
F1-score:  0.4891
ROC-AUC:   0.5244


(0.5205,
 0.518058690744921,
 0.46316851664984865,
 0.48907831646244004,
 0.5244224782207358)